```
brew install uv
uv vnev --python 3.12
uv pip install ipykernel pandas requests mecab gensim
```

In [1]:
import requests
import pandas as pd
import json

In [2]:
# OpenSearchの接続情報
BASE_URL = "http://localhost:9200"
INDEX_NAME = "ndl-bibliography"
HEADERS = {"Content-Type": "application/json"}

In [3]:
print(requests.get(f"{BASE_URL}/_cat/plugins").text)

ee9d0d1841a3 analysis-kuromoji                    3.7.0
ee9d0d1841a3 opensearch-alerting                  3.7.0.0
ee9d0d1841a3 opensearch-anomaly-detection         3.7.0.0
ee9d0d1841a3 opensearch-asynchronous-search       3.7.0.0
ee9d0d1841a3 opensearch-cross-cluster-replication 3.7.0.0
ee9d0d1841a3 opensearch-custom-codecs             3.7.0.0
ee9d0d1841a3 opensearch-flow-framework            3.7.0.0
ee9d0d1841a3 opensearch-geospatial                3.7.0.0
ee9d0d1841a3 opensearch-index-management          3.7.0.0
ee9d0d1841a3 opensearch-job-scheduler             3.7.0.0
ee9d0d1841a3 opensearch-knn                       3.7.0.0
ee9d0d1841a3 opensearch-ltr                       3.7.0.0
ee9d0d1841a3 opensearch-ml                        3.7.0.0
ee9d0d1841a3 opensearch-neural-search             3.7.0.0
ee9d0d1841a3 opensearch-notifications             3.7.0.0
ee9d0d1841a3 opensearch-notifications-core        3.7.0.0
ee9d0d1841a3 opensearch-observability             3.7.0.0
ee9d0d1841a3 ope

## schema(mappingの定義)

In [ ]:
mapping = {
    "settings": {
        "analysis": {
            "tokenizer": {"kuromoji_tokenizer": {"type": "kuromoji_tokenizer"}},
            "analyzer": {
                "kuromoji_analyzer": {
                    "type": "custom",
                    "tokenizer": "kuromoji_tokenizer",
                }
            },
        }
    },
    "mappings": {
        "properties": {
            "永続的識別子": {"type": "keyword"},
            "タイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "巻次又は部編番号": {"type": "keyword"},
            "シリーズタイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "版表示": {"type": "keyword"},
            "著者": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "出版者": {"type": "keyword"},
            "出版日": {"type": "keyword"},
            "出版日（W3CDTF）": {
                "type": "date",
                "format": "yyyy||yyyy-MM||yyyy-MM-dd||strict_date_optional_time||epoch_millis",
            },
            "識別子（ISBN）": {"type": "keyword"},
            "容量・大きさ": {"type": "text"},
            "分類（NDC）": {"type": "keyword"},
            "分類（NDC8）": {"type": "keyword"},
            "分類（NDC9）": {"type": "keyword"},
            "分類（NDLC）": {"type": "keyword"},
            "件名（NDLSH）": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "公開範囲": {"type": "keyword"},
        }
    },
}

In [ ]:
res = requests.put(
    f"{BASE_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping)
)
res.json()

In [ ]:
# requests.delete(f"{BASE_URL}/{INDEX_NAME}")

## Bulk APIを用いたfeed

In [ ]:
# df = pd.read_csv("../head.tsv", sep="\t")
df = pd.concat(
    [
        pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),
    ]
)
df.head(3)

In [ ]:
actions = df["永続的識別子"].map(
    lambda x: json.dumps(
        {
            "index": {
                "_index": INDEX_NAME,
                "_id": x.translate(str.maketrans({":": "-", "/": "-"})),
            }
        }
    )
)
data_jsons = df.to_json(orient="records", lines=True, force_ascii=False).splitlines()

jsonls = [
    action + "\n" + data_json + "\n" for action, data_json in zip(actions, data_jsons)
]

In [ ]:
# 10000件ずつ送信
chunk_size = 10000
for i in range(0, len(jsonls), chunk_size):
    print(f"Sending chunk {i} ~ {i + chunk_size}")
    jsonls_chunk = jsonls[i : i + chunk_size]
    print(jsonls_chunk[:3])

    res = requests.post(
        f"{BASE_URL}/{INDEX_NAME}/_bulk", headers=HEADERS, data="".join(jsonls_chunk)
    )
    res.raise_for_status()

In [ ]:
# # 削除
# requests.post(f"{BASE_URL}/{INDEX_NAME}/_delete_by_query", headers=HEADERS, data=json.dumps(
# {
#   "query": {
#     "match_all": {}
#   }
# })).json()

In [ ]:
count_res = requests.get(f"{BASE_URL}/{INDEX_NAME}/_count", headers=HEADERS)
print("=== _count API ===")
print(f"総ドキュメント数: {count_res.json()['count']}")
print(count_res.json())

In [ ]:
requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search",
    headers=HEADERS,
    data=json.dumps({"query": {"match_all": {}}}),
).json()

## 検索

In [51]:
query = {"query": {"match": {"著者": "東野圭吾"}}}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

{'took': 140,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 35, 'relation': 'eq'},
  'max_score': 10.687419,
  'hits': [{'_index': 'ndl-bibliography',
    '_id': 'info-ndljp-pid-12478369',
    '_score': 10.687419,
    '_source': {'永続的識別子': 'info:ndljp/pid/12478369',
     'タイトル': '卒業 : 雪月花殺人ゲーム',
     '巻次又は部編番号': None,
     'シリーズタイトル': None,
     '版表示': None,
     '著者': '東野圭吾 著',
     '出版者': '講談社',
     '出版日': '1986.5',
     '出版日（W3CDTF）': '1986',
     '識別子（ISBN）': '4-06-202728-3',
     '容量・大きさ': '319p ; 20cm',
     '分類（NDC）': None,
     '分類（NDC8）': 913.6,
     '分類（NDC9）': None,
     '分類（NDLC）': 'KH139',
     '件名（NDLSH）': None,
     '公開範囲': '国立国会図書館内限定公開'}},
   {'_index': 'ndl-bibliography',
    '_id': 'info-ndljp-pid-12478360',
    '_score': 10.687419,
    '_source': {'永続的識別子': 'info:ndljp/pid/12478360',
     'タイトル': '学生街の殺人',
     '巻次又は部編番号': None,
     'シリーズタイトル': None,
     '版表示': None,
     '著者': '東野圭吾 著',
  

In [ ]:
requests.get(f"{BASE_URL}/{INDEX_NAME}/_search?q=*:*").json()

## featuresetの定義と利用


In [ ]:
data = {"mappings": {"properties": {}}}
res = requests.put(f"{BASE_URL}/.ltrstore", headers=HEADERS, data=json.dumps(data))
print(res.json())

In [ ]:
data = {
    "featureset": {
        "features": [
            {
                "name": "title_match",
                "params": ["keywords"],
                "template": {"match": {"タイトル": "{{keywords}}"}},
            },
            {
                "name": "author_match",
                "params": ["keywords"],
                "template": {"match": {"著者": "{{keywords}}"}},
            },
            {
                "name": "publisher_match",
                "params": ["keywords"],
                "template": {"match": {"出版者": "{{keywords}}"}},
            },
        ]
    }
}

requests.post(
    f"{BASE_URL}/_ltr/_featureset/sample_featureset",
    data=json.dumps(data),
    headers=HEADERS,
).json()

In [ ]:
requests.get(f"{BASE_URL}/_ltr/_featureset").json()

## 特徴量をエンジンから計算

In [ ]:
query = {
    "query": {
        "bool": {
            "filter": [
                {
                    "terms": {
                        "_id": [
                            "info-ndljp-pid-12478369",
                            "info-ndljp-pid-13256368",
                        ]
                    }
                },
                {
                    "sltr": {
                        "_name": "logged_featureset",
                        "featureset": "sample_featureset",
                        "params": {"keywords": "東野圭吾"},
                    }
                },
            ]
        }
    },
    "ext": {
        "ltr_log": {
            "log_specs": {"name": "log_entry1", "named_query": "logged_featureset"}
        }
    },
}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

In [ ]:
query = {
    "size": 10,
    "query": {
        "bool": {
            "must": [{"match": {"著者": {"query": "東野圭吾", "operator": "and"}}}],
            "filter": [
                {
                    "sltr": {
                        "_name": "logged_featureset",
                        "featureset": "sample_featureset",
                        "params": {"keywords": "東野圭吾"},
                    }
                }
            ],
        }
    },
    "ext": {
        "ltr_log": {
            "log_specs": {"name": "log_entry1", "named_query": "logged_featureset"}
        }
    },
}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

## モデルのデプロイ
- いったん雑線形モデルから

In [5]:
# "model_type" ではなく "type" を使う必要があります

query = {
    "model": {
        "name": "sample_linear_model",
        "model": {
            "type": "model/linear",
            "definition": json.dumps(
                {"title_match": 0.2, "author_match": 0.1, "publisher_match": 0.1}
            ),
        },
    }
}

endpoint = f"{BASE_URL}/_ltr/_featureset/sample_featureset/_createmodel"
res = requests.post(endpoint, data=json.dumps(query), headers=HEADERS)
res.json()

{'error': {'root_cause': [{'type': 'illegal_argument_exception',
    'reason': 'Element of type [model] are not updatable, please create a new one instead.'}],
  'type': 'illegal_argument_exception',
  'reason': 'Element of type [model] are not updatable, please create a new one instead.',
  'suppressed': [{'type': 'version_conflict_engine_exception',
    'reason': '[model-sample_linear_model]: version conflict, document already exists (current version [3])',
    'index': '.ltrstore',
    'shard': '0',
    'index_uuid': 'uvJyswt-TjCTS2EgYm1plw'}]},
 'status': 405}

In [ ]:
# delete_endpoint = f"{BASE_URL}/_ltr/_model/sample_linear_model"
# res = requests.delete(delete_endpoint, headers=HEADERS)
# res.json()

In [ ]:
requests.get(f"{BASE_URL}/_ltr/_model").json()

## ベクトル検索の実装

In [9]:
import MeCab
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [5]:
# データ読み込み
# df = pd.read_csv("../head.tsv", sep="\t")
df = pd.concat(
    [
        pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),
    ]
)

/tmp/ipykernel_7996/3411647010.py:5: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
/tmp/ipykernel_7996/3411647010.py:6: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
/tmp/ipykernel_7996/3411647010.py:7: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),


In [13]:
# 日本語テキスト前処理（最小化版）
import re
import pandas as pd
import MeCab


def preprocess_japanese_text(text):
    """日本語テキストの前処理"""
    if pd.isna(text) or text == "":
        return []

    tagger = MeCab.Tagger("-r /etc/mecabrc -Owakati")
    text_cleaned = re.sub(
        r"[^\w\s\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]", " ", str(text)
    )
    result = tagger.parse(text_cleaned).strip()
    words = result.split()
    return [word for word in words if len(word) > 1]


# テスト実行
test_text = "吾輩は猫である。名前はまだ無い。"
print(f"テスト入力: {test_text}")
result = preprocess_japanese_text(test_text)
print(f"分かち書き結果: {result}")

テスト入力: 吾輩は猫である。名前はまだ無い。
分かち書き結果: ['吾輩', 'ある', '名前', 'まだ', '無い']


In [7]:
# データの準備とWord2Vecモデルの訓練
print("=== テキストデータの準備 ===")

# タイトルと著者のテキストを結合
df["combined_text"] = df["タイトル"].fillna("") + " " + df["著者"].fillna("")

# 各行のテキストを形態素解析
print("形態素解析実行中...")
sentences = []
for idx, text in enumerate(df["combined_text"]):
    words = preprocess_japanese_text(text)
    if words:  # 空でない場合のみ追加
        sentences.append(words)

print("サンプル文書:", sentences[:3])

=== テキストデータの準備 ===
形態素解析実行中...
サンプル文書: [['美術', '展覧', '記念', '画集', '広陵', '美術'], ['相撲', '大観', '三木', '愛花', '山田', '春塘', '共編'], ['大阪', '労働', '組合', '名簿']]


In [8]:
# Word2Vecモデルの訓練
print("\n=== Word2Vecモデル訓練 ===")

# Word2Vecのパラメータ
vector_size = 128  # ベクトルの次元数
window = 5  # 文脈窓のサイズ
min_count = 2  # 最小出現回数
workers = 4  # 並列処理数

# モデル訓練
print("Word2Vecモデル訓練中...")
model = Word2Vec(
    sentences=sentences,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    workers=workers,
    epochs=10,
)

print(f"語彙数: {len(model.wv.key_to_index)}")
print("訓練完了")

# モデルを保存
model.save("../data/word2vec_model.model")
print(f"モデルを保存しました")

# モデルテスト
test_words = ["東野", "圭吾", "小説", "推理"]
for word in test_words:
    if word in model.wv:
        similar_words = model.wv.most_similar(word, topn=5)
        print(f"'{word}'に類似する語: {similar_words}")


=== Word2Vecモデル訓練 ===
Word2Vecモデル訓練中...
語彙数: 61840
訓練完了
モデルを保存しました
'東野'に類似する語: [('黙阿弥', 0.8330104351043701), ('河竹', 0.8262487053871155), ('愛宕', 0.8045657277107239), ('峯岸', 0.7950685024261475), ('須磨', 0.7908071279525757)]
'圭吾'に類似する語: [('Borgford', 0.7731873393058777), ('仮名書', 0.7360553741455078), ('俳文', 0.7236427068710327), ('放電', 0.7144998908042908), ('能登半島', 0.7071027755737305)]
'小説'に類似する語: [('推理', 0.6988710165023804), ('長編', 0.6868109107017517), ('伝奇', 0.6726084351539612), ('長篇', 0.6492440700531006), ('ネビロス', 0.6442238688468933)]
'推理'に類似する語: [('ミステリー', 0.8514480590820312), ('本格', 0.8255556225776672), ('長編', 0.8016732931137085), ('サスペンス', 0.7808302044868469), ('長篇', 0.7621681094169617)]


In [10]:
model = Word2Vec.load("../data/word2vec_model.model")

In [14]:
# AvgPoolingベースのベクトル化関数
def text_to_vector(text, model, vector_size=128) -> np.ndarray | None:
    """
    テキストをWord2VecのAvgPoolingでベクトル化する関数。

    - 入力:
        - text: ベクトル化対象のテキスト（日本語）。
        - model: Word2Vecモデル。
        - vector_size: ベクトルの次元数（デフォルト128）。
    - 処理:
        - テキストを形態素解析して単語リストに変換。
        - Word2Vecモデルに存在する単語のベクトルを取得。
        - 単語ベクトルの平均を計算して文書ベクトルを生成。
    - 出力:
        - 文書ベクトル（numpy配列）。
    """
    words = preprocess_japanese_text(text)
    if not words:
        return None

    # 語彙に存在する単語のベクトルを取得
    word_vectors = []
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])

    if not word_vectors:
        return None

    # 平均プーリング
    return np.mean(word_vectors, axis=0)


# テスト実行
print("\n=== ベクトル化テスト ===")
test_docs = [
    "東野圭吾 容疑者Xの献身",
    "村上春樹 ノルウェイの森",
    "夏目漱石 吾輩は猫である",
]

vectors = []
for doc in test_docs:
    vec = text_to_vector(doc, model)
    vectors.append(vec)
    print(f"'{doc}' -> ベクトル形状: {vec.shape}, 最初の5要素: {vec[:5]}")

# ベクトル間の類似度計算
vectors = np.array(vectors)
similarity_matrix = cosine_similarity(vectors)
print(f"\n類似度行列:\n{similarity_matrix}")


=== ベクトル化テスト ===
'東野圭吾 容疑者Xの献身' -> ベクトル形状: (128,), 最初の5要素: [ 0.02566626 -0.01389022  0.05475742  0.08650817  0.16365664]
'村上春樹 ノルウェイの森' -> ベクトル形状: (128,), 最初の5要素: [ 0.16932873 -0.5699959   0.1292599   0.5160305  -0.06183125]
'夏目漱石 吾輩は猫である' -> ベクトル形状: (128,), 最初の5要素: [ 0.93053484 -0.3029918   0.91771257  0.05476594  0.3987335 ]

類似度行列:
[[1.0000001  0.3763245  0.3027275 ]
 [0.3763245  1.0000002  0.19395998]
 [0.3027275  0.19395998 1.0000002 ]]


## OpenSearchベクトル検索の実装
先ほど作ったKW用indexとは異なるindexを作成してそちらで検索などする

In [6]:
VECTOR_INDEX_NAME = "ndl-bibliography-vector"

In [58]:
# ベクトル検索用マッピング
vector_mapping = {
    "settings": {
        "analysis": {
            "tokenizer": {"kuromoji_tokenizer": {"type": "kuromoji_tokenizer"}},
            "analyzer": {
                "kuromoji_analyzer": {
                    "type": "custom",
                    "tokenizer": "kuromoji_tokenizer",
                }
            },
        },
        "index.knn": True,
    },
    "mappings": {
        "properties": {
            "永続的識別子": {"type": "keyword"},
            "タイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "著者": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "出版者": {"type": "keyword"},
            "出版日": {"type": "keyword"},
            "出版日（W3CDTF）": {
                "type": "date",
                "format": "yyyy||yyyy-MM||yyyy-MM-dd||strict_date_optional_time||epoch_millis",
            },
            "combined_text": {"type": "text", "analyzer": "kuromoji_analyzer"},
            # ベクトルフィールド
            "title_author_vector": {
                "type": "knn_vector",
                "dimension": 128,
                "method": {
                    "name": "hnsw",  # Hierarchical Navigable Small World
                    "space_type": "cosinesimil",  # コサイン類似度
                    "engine": "faiss",
                    "parameters": {
                        "ef_construction": 256,  # インデックス構築時のef値
                        "m": 16,  # HNSWの接続数
                    },
                },
            },
        }
    },
}

# 既存のベクトルインデックスを削除（存在する場合）
print("=== ベクトルインデックス作成 ===")
delete_res = requests.delete(f"{BASE_URL}/{VECTOR_INDEX_NAME}", headers=HEADERS)
print(f"既存インデックス削除: {delete_res.status_code}")

# 新しいインデックスを作成
create_res = requests.put(
    f"{BASE_URL}/{VECTOR_INDEX_NAME}", headers=HEADERS, data=json.dumps(vector_mapping)
)
print(f"インデックス作成: {create_res.status_code}")
create_result = create_res.json()
print(create_result)

if create_result.get("acknowledged"):
    print("✅ ベクトル検索用インデックス作成完了")
else:
    print("❌ インデックス作成失敗")

=== ベクトルインデックス作成 ===
既存インデックス削除: 200
インデックス作成: 200
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'ndl-bibliography-vector'}
✅ ベクトル検索用インデックス作成完了


In [59]:
# OpenSearchへのベクトルデータ投入
print("\n=== OpenSearchへのベクトルデータ投入 ===")

# Bulk APIデータ準備

test_df = df.reset_index(drop=True).copy()

# Bulk投入実行
print("Bulk投入実行中...")
chunk_size = 1000
for i in range(0, len(test_df), chunk_size):
    print(f"投入中: {i} ~ {i + chunk_size}")
    test_df_sclice = test_df.iloc[i : i + chunk_size].reset_index(drop=True).copy()

    # ベクトル化実行
    test_df_sclice["title_author_vector"] = test_df_sclice["combined_text"].apply(
        lambda x: (
            arr.tolist()
            if (arr := text_to_vector(x, model, vector_size)) is not None
            else None
        )
    )

    # feed用のデータ整形
    vector_actions = test_df_sclice["永続的識別子"].map(
        lambda x: json.dumps(
            {
                "index": {
                    "_index": VECTOR_INDEX_NAME,
                    "_id": x.translate(str.maketrans({":": "-", "/": "-"})),
                }
            }
        )
    )

    # データをJSON形式に変換（ベクトルを含む）
    vector_data_jsons = test_df_sclice.to_json(
        orient="records", lines=True, force_ascii=False
    ).splitlines()

    vector_jsonls = [
        action + "\n" + data_json + "\n"
        for action, data_json in zip(vector_actions, vector_data_jsons)
    ]

    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_bulk",
        headers=HEADERS,
        data="".join(vector_jsonls),
    )

    bulk_result = res.json()
    if bulk_result.get("errors"):
        print("Bulkエラーが発生:")
        for item in bulk_result.get("items", []):  # 最初の3つのエラーを表示
            if "index" in item and item["index"]["status"] not in (200, 201):
                print(item)

print("投入完了")

# ドキュメント数確認
count_res = requests.get(f"{BASE_URL}/{VECTOR_INDEX_NAME}/_count", headers=HEADERS)
print(f"投入されたドキュメント数: {count_res.json()['count']}")


=== OpenSearchへのベクトルデータ投入 ===
Bulk投入実行中...
投入中: 0 ~ 1000
投入中: 1000 ~ 2000
投入中: 2000 ~ 3000
投入中: 3000 ~ 4000
投入中: 4000 ~ 5000
投入中: 5000 ~ 6000
投入中: 6000 ~ 7000
投入中: 7000 ~ 8000
投入中: 8000 ~ 9000
投入中: 9000 ~ 10000
投入中: 10000 ~ 11000
投入中: 11000 ~ 12000
投入中: 12000 ~ 13000
投入中: 13000 ~ 14000
Bulkエラーが発生:
{'index': {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13317728', 'status': 400, 'error': {'type': 'mapper_parsing_exception', 'reason': "failed to parse field [出版日（W3CDTF）] of type [date] in document with id 'info-ndljp-pid-13317728'. Preview of field's value: '1978||1994'", 'caused_by': {'type': 'illegal_argument_exception', 'reason': 'failed to parse date field [1978||1994] with format [yyyy||yyyy-MM||yyyy-MM-dd||strict_date_optional_time||epoch_millis]', 'caused_by': {'type': 'date_time_parse_exception', 'reason': 'Failed to parse with all enclosed parsers'}}}}}
{'index': {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13317602', 'status': 400, 'error': {'typ

In [16]:
# ベクトル検索機能の実装
def vector_search(query_text, k=5):
    """
    クエリテキストに基づくベクトル検索
    """
    # クエリをベクトル化
    query_vector = text_to_vector(query_text, model)

    # OpenSearchでベクトル検索
    search_query = {
        "size": k,
        "_source": ["タイトル", "著者", "出版者", "出版日"],
        "query": {
            "knn": {"title_author_vector": {"vector": query_vector.tolist(), "k": k}}
        },
    }

    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_search",
        headers=HEADERS,
        data=json.dumps(search_query),
    )

    return res.json()


# ベクトル検索テスト
print("\n=== ベクトル検索テスト ===")

test_queries = ["東野圭吾 推理小説", "村上春樹 恋愛", "夏目漱石 古典文学"]

for query in test_queries:
    print(f"\n検索クエリ: '{query}'")
    print(vector_search(query, k=5))


=== ベクトル検索テスト ===

検索クエリ: '東野圭吾 推理小説'
{'took': 708, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 0.99152315, 'hits': [{'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-1671413', '_score': 0.99152315, '_source': {'出版者': '中央公論社', '出版日': '1962', 'タイトル': '方壷園 : 推理小説集', '著者': '陳舜臣 著'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-12562831', '_score': 0.99096525, '_source': {'出版者': '講談社', '出版日': '1977.5', 'タイトル': '闇の金魚 : 推理小説', '著者': '陳舜臣 著'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-12493390', '_score': 0.98756534, '_source': {'出版者': '光文社', '出版日': '1979.1', 'タイトル': '誰が引金をひいた : 長編推理小説', '著者': '菊村到 著'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-1358497', '_score': 0.98703486, '_source': {'出版者': '五月書房', '出版日': '1959', 'タイトル': '花火 : 長編推理小説', '著者': '葉室早生 著'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-1

In [ ]:
# ハイブリッド検索（キーワード検索 + ベクトル検索）
def hybrid_search(query_text, keyword_weight=0.5, vector_weight=0.5, k=5):
    """
    キーワード検索とベクトル検索を組み合わせたハイブリッド検索
    """
    # クエリをベクトル化
    query_vector = text_to_vector(query_text, model)

    # ハイブリッド検索クエリ
    search_query = {
        "size": k,
        "_source": ["タイトル", "著者", "出版者", "出版日"],
        "query": {
            "bool": {
                "should": [
                    # キーワード検索部分
                    {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["タイトル^2", "著者^1.5", "combined_text"],
                            "type": "best_fields",
                            # keyword weightはhybrid検索でのスコアに対する重みを指定している。
                            "boost": keyword_weight,
                        }
                    },
                    # ベクトル検索部分
                    {
                        "knn": {
                            "title_author_vector": {
                                "vector": query_vector.tolist(),
                                "k": k * 2,  # より多くの候補を取得
                                "boost": vector_weight,
                            }
                        }
                    },
                ]
            }
        },
    }

    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_search",
        headers=HEADERS,
        data=json.dumps(search_query),
    )

    return res.json()


# ハイブリッド検索テスト
print("\n=== ハイブリッド検索テスト ===")

test_query = "イカゲーム"
print(f"検索クエリ: '{test_query}'")
print("=" * 60)

# 3つの検索方法を比較
print("1. キーワード検索のみ")
print("-" * 30)
keyword_result = hybrid_search(test_query, keyword_weight=1.0, vector_weight=0.0, k=3)
if "hits" in keyword_result and keyword_result["hits"]["hits"]:
    for i, hit in enumerate(keyword_result["hits"]["hits"], 1):
        source = hit["_source"]
        score = hit["_score"]
        print(
            f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}"
        )

print("\n2. ベクトル検索のみ")
print("-" * 30)
vector_result = hybrid_search(test_query, keyword_weight=0.0, vector_weight=1.0, k=3)
if "hits" in vector_result and vector_result["hits"]["hits"]:
    for i, hit in enumerate(vector_result["hits"]["hits"], 1):
        source = hit["_source"]
        score = hit["_score"]
        print(
            f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}"
        )

print("\n3. ハイブリッド検索（キーワード50% + ベクトル50%）")
print("-" * 30)
hybrid_result = hybrid_search(test_query, keyword_weight=0.5, vector_weight=0.5, k=3)
if "hits" in hybrid_result and hybrid_result["hits"]["hits"]:
    for i, hit in enumerate(hybrid_result["hits"]["hits"], 1):
        source = hit["_source"]
        score = hit["_score"]
        print(
            f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}"
        )


=== ハイブリッド検索テスト ===
検索クエリ: 'イカゲーム'
1. キーワード検索のみ
------------------------------
1. スコア: 11.5069 | 日本海のイカ - 足立倫行 著
2. スコア: 10.6250 | 八戸のイカ釣 - 八戸市博物館 編
3. スコア: 10.6250 | イカしてソーロウ - 笑太郎 著

2. ベクトル検索のみ
------------------------------
1. スコア: 0.9908 | インスピレーション・ゲーム - となみむか 著
2. スコア: 0.9878 | 男と女のラブゲーム - 岩月謙司 著
3. スコア: 0.9851 | りびんぐゲーム - 星里もちる 著

3. ハイブリッド検索（キーワード50% + ベクトル50%）
------------------------------
1. スコア: 5.7535 | 日本海のイカ - 足立倫行 著
2. スコア: 5.3125 | 八戸のイカ釣 - 八戸市博物館 編
3. スコア: 5.3125 | イカしてソーロウ - 笑太郎 著


In [ ]:
test_query = "村上春樹"

keyword_result = hybrid_search(test_query, keyword_weight=1.0, vector_weight=0.0, k=100)
if "hits" in keyword_result and keyword_result["hits"]["hits"]:
    for i, hit in enumerate(keyword_result["hits"]["hits"], 1):
        source = hit["_source"]
        score = hit["_score"]
        print(
            f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}"
        )

1. スコア: 18.9675 | ウォーク・ドント・ラン : 村上龍vs村上春樹 - 村上龍, 村上春樹 著
2. スコア: 11.1037 | 平将門伝説一覧 - 村上春樹 編
3. スコア: 11.1037 | ノルウェイの森 - 村上春樹 著
4. スコア: 11.1037 | ノルウェイの森 - 村上春樹 著
5. スコア: 11.1037 | 蛍・納屋を焼く・その他の短編 - 村上春樹 著
6. スコア: 11.1037 | The scrap : 懐かしの一九八〇年代 - 村上春樹 著
7. スコア: 11.1037 | カンガルー日和 - 村上春樹 [著]
8. スコア: 11.1037 | パン屋再襲撃 - 村上春樹 著
9. スコア: 11.1037 | 中国行きのスロウ・ボート - 村上春樹 著
10. スコア: 11.1037 | 羊をめぐる冒険 - 村上春樹 [著]
11. スコア: 11.1037 | 羊をめぐる冒険 - 村上春樹 [著]
12. スコア: 11.1037 | 回転木馬のデッド・ヒート - 村上春樹 著
13. スコア: 11.1037 | 中国行きのスロウ・ボート - 村上春樹 著
14. スコア: 11.1037 | 世界の終りとハードボイルド・ワンダーランド - 村上春樹 著
15. スコア: 11.1037 | 螢・納屋を焼く・その他の短編 - 村上春樹 著
16. スコア: 11.1037 | カンガルー日和 - 村上春樹 著
17. スコア: 11.1037 | 1973年のピンボール - 村上春樹 [著]
18. スコア: 11.1037 | 羊をめぐる冒険 - 村上春樹 著
19. スコア: 11.1037 | 風の歌を聴け - 村上春樹 [著]
20. スコア: 11.1037 | 1973年のピンボール - 村上春樹 著
21. スコア: 11.1037 | 風の歌を聴け - 村上春樹 著
22. スコア: 10.9122 | 角川春樹集・猿田彦 - None
23. スコア: 10.8391 | 村上城跡 - 市原市文化財センター 編
24. スコア: 10.8391 | 村上華岳 - 河北倫明 著
25. スコア: 9.9394 | 村上朝日堂 - 村上春樹, 安西水丸 著
26. スコア: 9.9